In [88]:
# 1. Import Library
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report)

In [89]:
# 2. Membaca Dataset
from pathlib import Path

file_path = Path("healthcare-dataset-stroke-data.csv")
df = pd.read_csv(file_path)

print("Ukuran dataset:", df.shape)
display(df.head())


Ukuran dataset: (5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [90]:
# 3.Info dan Tipe Kolom

print("Informasi dataset:")
df.info()

Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [91]:
# 4. Describe
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,5110.0,36517.829354,21161.721625,67.00,17741.250,36932.000,54682.00,72940.00
age,5110.0,43.226614,22.612647,0.08,25.000,45.000,61.00,82.00
hypertension,5110.0,0.097456,0.296607,0.00,0.000,0.000,0.00,1.00
heart_disease,5110.0,0.054012,0.226063,0.00,0.000,0.000,0.00,1.00
avg_glucose_level,5110.0,106.147677,45.283560,55.12,77.245,91.885,114.09,271.74
bmi,4909.0,28.893237,7.854067,10.30,23.500,28.100,33.10,97.60
stroke,5110.0,0.048728,0.215320,0.00,0.000,0.000,0.00,1.00


In [92]:
# 5. Koreksi Tipe Data
kolom_kategorikal = [
    "gender", "ever_married", "work_type",
    "Residence_type", "smoking_status"]

df[kolom_kategorikal] = df[kolom_kategorikal].astype("category")

print("Tipe data setelah koreksi:")
print(df.dtypes)

Tipe data setelah koreksi:
id                      int64
gender               category
age                   float64
hypertension            int64
heart_disease           int64
ever_married         category
work_type            category
Residence_type       category
avg_glucose_level     float64
bmi                   float64
smoking_status       category
stroke                  int64
dtype: object


In [93]:
# 6. Missing Value 
print('Nilai kosong per kolom:')
df.isna().sum()

Nilai kosong per kolom:


id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [94]:
# 7 Penanganan Missing Value

median_bmi = df['bmi'].median()
df['bmi'] = df['bmi'].fillna(median_bmi)

# Memeriksa kembali missing value
print(df.isnull().sum())

id                   0
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64


In [95]:
# 8. Duplikasi
jumlah_duplikat = df.drop(columns=['id'], errors='ignore').duplicated().sum()
print('Duplikat (tanpa menghitung kolom id):', jumlah_duplikat)

if jumlah_duplikat > 0:
    df = df.drop_duplicates(subset=[c for c in df.columns if c != 'id']).copy()

print('Ukuran data setelah pemeriksaan:', df.shape)

Duplikat (tanpa menghitung kolom id): 0
Ukuran data setelah pemeriksaan: (5110, 12)


In [96]:
# 9. Normalisasi dan Standarisasi
from sklearn.preprocessing import MinMaxScaler, StandardScaler

kolom_norm = ["age", "avg_glucose_level", "bmi"]
kolom_std = ["age", "avg_glucose_level", "bmi"]

df_hasil = df.copy()

# Normalisasi Min-Max (skala 0-1)
df_hasil[kolom_norm] = MinMaxScaler().fit_transform(
    df_hasil[kolom_norm])

# Standarisasi Z-Score
df_hasil[kolom_std] = StandardScaler().fit_transform(
    df_hasil[kolom_std])

print("Nilai minimum dan maksimum setelah normalisasi dan standarisasi:")
print(df_hasil[kolom_norm].agg(["min", "max"]).round(3))

print("Rata-rata dan standar deviasi setelah standarisasi:")
print(df_hasil[kolom_std].agg(["mean", "std"]).round(3))

print("Bentuk data hasil pemrosesan:", df_hasil.shape)

Nilai minimum dan maksimum setelah normalisasi dan standarisasi:
       age  avg_glucose_level    bmi
min -1.908             -1.127 -2.411
max  1.715              3.657  8.928
Rata-rata dan standar deviasi setelah standarisasi:
      age  avg_glucose_level  bmi
mean -0.0               -0.0 -0.0
std   1.0                1.0  1.0
Bentuk data hasil pemrosesan: (5110, 12)


In [97]:
# 10. Kategori (ejaan dan kapitalisasi)

kolom_kategori = df.select_dtypes(
    include=['object', 'category', 'string']).columns.tolist()

print('Kolom kategori:', kolom_kategori)

for kolom in kolom_kategori:
    print(f'Kategori pada kolom {kolom}:')
    print(sorted(df[kolom].dropna().astype(str).unique()))

    # Cek perbedaan kapitalisasi atau spasi
    kelompok = {}
    for nilai in df[kolom].dropna().astype(str).unique():
        kunci = nilai.strip().casefold()
        kelompok.setdefault(kunci, []).append(nilai)

    for variasi in kelompok.values():
        if len(variasi) > 1:
            print('Perlu diperiksa (ejaan/kapitalisasi):', variasi)


Kolom kategori: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
Kategori pada kolom gender:
['Female', 'Male', 'Other']
Kategori pada kolom ever_married:
['No', 'Yes']
Kategori pada kolom work_type:
['Govt_job', 'Never_worked', 'Private', 'Self-employed', 'children']
Kategori pada kolom Residence_type:
['Rural', 'Urban']
Kategori pada kolom smoking_status:
['Unknown', 'formerly smoked', 'never smoked', 'smokes']


In [98]:
# 11. Memeriksa Outlier menggunakan IQR

for kolom in ['age', 'avg_glucose_level', 'bmi']:
    nilai = df[kolom].dropna()
    q1 = nilai.quantile(0.25)
    q3 = nilai.quantile(0.75)
    iqr = q3 - q1
    batas_bawah = q1 - 1.5 * iqr
    batas_atas = q3 + 1.5 * iqr
    jumlah_outlier = ((nilai < batas_bawah) | (nilai > batas_atas)).sum()
    print(f'{kolom}: jumlah outlier = {jumlah_outlier}; batas = ({batas_bawah:.2f}, {batas_atas:.2f})')

age: jumlah outlier = 0; batas = (-29.00, 115.00)
avg_glucose_level: jumlah outlier = 627; batas = (21.98, 169.36)
bmi: jumlah outlier = 126; batas = (10.30, 46.30)


In [99]:
# 12. Penanganan Outlier

for kolom in ['age', 'avg_glucose_level', 'bmi']:
    nilai = df[kolom].dropna()

    q1 = nilai.quantile(0.25)
    q3 = nilai.quantile(0.75)
    iqr = q3 - q1

    batas_bawah = q1 - 1.5 * iqr
    batas_atas = q3 + 1.5 * iqr

    # Mengganti nilai outlier dengan nilai batas
    df[kolom] = df[kolom].clip(
        lower=batas_bawah,
        upper=batas_atas
    )

# Menampilkan statistik setelah penanganan outlier
print("Statistik data setelah penanganan outlier:")
print(df[['age', 'avg_glucose_level', 'bmi']].describe())


Statistik data setelah penanganan outlier:
               age  avg_glucose_level          bmi
count  5110.000000        5110.000000  5110.000000
mean     43.226614         100.996204    28.690411
std      22.612647          33.214738     7.120858
min       0.080000          55.120000    10.300000
25%      25.000000          77.245000    23.800000
50%      45.000000          91.885000    28.100000
75%      61.000000         114.090000    32.800000
max      82.000000         169.357500    46.300000


In [100]:
# 13. Ketidakseimbangan kelas

jumlah_kelas = df["stroke"].value_counts()
persentase_kelas = df["stroke"].value_counts(normalize=True) * 100

print("Penjelasan kelas target:")
print("0 = Tidak mengalami stroke")
print("1 = Mengalami stroke")

print("Jumlah:")
print(
    f"Kelas 0 (tidak mengalami stroke): "
    f"{jumlah_kelas.get(0, 0)} data "
    f"({persentase_kelas.get(0, 0):.2f}%).")
print(
    f"Kelas 1 (mengalami stroke): "
    f"{jumlah_kelas.get(1, 0)} data "
    f"({persentase_kelas.get(1, 0):.2f}%).")

Penjelasan kelas target:
0 = Tidak mengalami stroke
1 = Mengalami stroke
Jumlah:
Kelas 0 (tidak mengalami stroke): 4861 data (95.13%).
Kelas 1 (mengalami stroke): 249 data (4.87%).


In [101]:
# 14. Pemeriksaan Indikasi Kebocoran Data (Leakage)
TARGET = "stroke"

# Memisahkan fitur dan target
X = df.drop(columns=[TARGET, "id"], errors="ignore")
y = df[TARGET]

print("Apakah target masih ada di fitur?")
print(TARGET in X.columns)

# Korelasi fitur numerik dengan target
tmp = X.copy()
tmp["target_stroke"] = y
num_cols = tmp.select_dtypes(include="number").columns.drop(
    "target_stroke", errors="ignore")
print("\nKorelasi fitur numerik dengan target:")
print(
    tmp[list(num_cols) + ["target_stroke"]]
    .corr(numeric_only=True)["target_stroke"]
    .drop("target_stroke")
    .round(3))

# Memeriksa baris dengan fitur identik
duplikat_fitur = X.duplicated(keep=False).sum()
print("\nJumlah baris dengan fitur identik:", duplikat_fitur)

# Memeriksa fitur kategorikal
cat_cols = X.select_dtypes(include=["object", "category"]).columns
print("\nKategori unik setiap fitur kategorikal:")
for kolom in cat_cols:
    print(f"{kolom}: {X[kolom].nunique()} kategori")

# Menampilkan fitur akhir
print("\nFitur yang digunakan:")
print(X.columns.tolist())
print("Jumlah fitur:", X.shape[1])

Apakah target masih ada di fitur?
False

Korelasi fitur numerik dengan target:
age                  0.245
hypertension         0.128
heart_disease        0.135
avg_glucose_level    0.116
bmi                  0.043
Name: target_stroke, dtype: float64

Jumlah baris dengan fitur identik: 0

Kategori unik setiap fitur kategorikal:
gender: 3 kategori
ever_married: 2 kategori
work_type: 5 kategori
Residence_type: 2 kategori
smoking_status: 4 kategori

Fitur yang digunakan:
['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status']
Jumlah fitur: 10


In [102]:
# 15. Encoding

X = df.drop(columns=['id', 'stroke'])
y = df['stroke']

X = pd.get_dummies(X, drop_first=True, dtype=int)

display(X.head())
print("Jumlah fitur:", X.shape[1])

,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Male,gender_Other,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67.0,0,1,169.3575,36.6,1,0,1,0,1,0,0,1,1,0,0
1,61.0,0,0,169.3575,28.1,0,0,1,0,0,1,0,0,0,1,0
2,80.0,0,1,105.9200,32.5,1,0,1,0,1,0,0,0,0,1,0
3,49.0,0,0,169.3575,34.4,0,0,1,0,1,0,0,1,0,0,1
4,79.0,1,0,169.3575,24.0,0,0,1,0,0,1,0,0,0,1,0


Jumlah fitur: 16


In [103]:
# 16. 5 data teratas

print("5 Data Teratas:")
display(df.head(5))

5 Data Teratas:


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,169.3575,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,169.3575,28.1,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.9200,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,169.3575,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,169.3575,24.0,never smoked,1


In [104]:
# 17. 5 data terbawah

print("5 Data Terbawah:")
display(df.tail(5))

5 Data Terbawah:


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,28.1,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0
5109,44679,Female,44.0,0,0,Yes,Govt_job,Urban,85.28,26.2,Unknown,0


In [105]:
# 18. Gaussian Naive Bayes

# Menentukan fitur dan target
X = df.drop(columns=["id", "stroke"])
y = df["stroke"]

# Mengubah fitur kategorik menjadi numerik
X = pd.get_dummies(X, drop_first=True, dtype=int)

# Membagi data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Menangani nilai kosong berdasarkan data latih
median_train = X_train.median()

X_train = X_train.fillna(median_train)
X_test = X_test.fillna(median_train)

# Membuat dan melatih model
model = GaussianNB()
model.fit(X_train, y_train)

# Melakukan prediksi
y_pred = model.predict(X_test)

# Evaluasi model
print("Akurasi:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(
    y_test, y_pred, zero_division=0))

print("Distribusi data uji:")
print(y_test.value_counts())

print("Distribusi hasil prediksi:")
print(pd.Series(y_pred).value_counts())

print("Jumlah nilai kosong :")
print(X.isnull().sum().sum())

Akurasi: 40.12 %
Confusion Matrix:
[[361 611]
 [  1  49]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.37      0.54       972
           1       0.07      0.98      0.14        50

    accuracy                           0.40      1022
   macro avg       0.54      0.68      0.34      1022
weighted avg       0.95      0.40      0.52      1022

Distribusi data uji:
stroke
0    972
1     50
Name: count, dtype: int64
Distribusi hasil prediksi:
1    660
0    362
Name: count, dtype: int64
Jumlah nilai kosong :
0
